# 03 — RFM Scoring & Customer Segmentation

1. Build customer-level Recency / Frequency / Monetary table  
2. Score each dimension 1–5  
3. Map scores onto business segments  
4. Cluster with K-Means (Elbow + Silhouette)  
5. Compare Hierarchical and DBSCAN


In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import joblib

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.rfm import compute_rfm, score_rfm, segment_customers
from src.clustering import (
    preprocess_for_clustering,
    run_kmeans,
    elbow_method,
    silhouette_analysis,
    run_hierarchical,
    run_dbscan,
    attach_labels,
    cluster_profiles,
)

pd.set_option("display.float_format", "{:.2f}".format)

reg = pd.read_csv(ROOT / "data" / "registered_clean.csv", parse_dates=["InvoiceDate"])
print(f"Working with {reg['CustomerID'].nunique():,} registered customers")

Working with 4,338 registered customers


## 1. RFM table

- **Recency** = days since last invoice  
- **Frequency** = number of unique invoices  
- **Monetary** = sum of LineRevenue

In [12]:
rfm = compute_rfm(reg)
rfm[["Recency", "Frequency", "Monetary", "AvgOrderValue", "TenureDays"]].describe()

,Recency,Frequency,Monetary,AvgOrderValue,TenureDays
count,4338.00,4338.00,4338.00,4338.00,4338.00
mean,92.54,4.27,2054.27,419.17,130.45
std,100.01,7.70,8989.23,1796.54,132.04
min,1.00,1.00,3.75,3.45,0.00
25%,18.00,1.00,307.41,178.62,0.00
50%,51.00,2.00,674.49,293.90,92.50
75%,142.00,5.00,1661.74,430.11,251.75
max,374.00,209.00,280206.02,84236.25,373.00


## 2. Quantile scoring (1–5)

Recency is reversed so that a score of 5 always means “best”.

In [13]:
rfm = score_rfm(rfm)
print(rfm[["R_Score", "F_Score", "M_Score", "RFM_Score", "RFM_Sum"]].head(8))
print("\nScore distributions:")
for col in ["R_Score", "F_Score", "M_Score"]:
    print(col, rfm[col].value_counts().sort_index().to_dict())

   R_Score  F_Score  M_Score RFM_Score  RFM_Sum
0        1        1        5       115        7
1        5        5        5       555       15
2        2        4        4       244       10
3        4        1        4       414        9
4        1        1        2       112        4
5        3        5        5       355       13
6        1        1        1       111        3
7        1        1        4       114        6

Score distributions:
R_Score {1: 865, 2: 843, 3: 858, 4: 904, 5: 868}
F_Score {1: 868, 2: 867, 3: 868, 4: 867, 5: 868}
M_Score {1: 868, 2: 867, 3: 868, 4: 867, 5: 868}


## 3. Business segments

In [14]:
rfm = segment_customers(rfm)
print(rfm["Segment"].value_counts())

seg_profile = (
    rfm.groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        AvgR=("Recency", "mean"),
        AvgF=("Frequency", "mean"),
        AvgM=("Monetary", "mean"),
        TotalRev=("Monetary", "sum"),
    )
    .round(1)
    .sort_values("TotalRev", ascending=False)
)
seg_profile

Segment
Others                        909
Lost / Hibernating            824
Need Attention                718
Loyal Customers               614
Champions                     348
Potential Loyalists           337
New Inflow                    234
Promising                     188
At-Risk / High-Value Churn    166
Name: count, dtype: int64


,Customers,AvgR,AvgF,AvgM,TotalRev
Segment,,,,,
Champions,348,5.40,18.20,11221.70,3905165.90
Loyal Customers,614,17.10,7.00,3101.30,1904175.20
Others,909,45.90,3.20,1158.80,1053306.30
Need Attention,718,168.70,2.20,1059.00,760329.20
Potential Loyalists,337,16.50,3.20,1434.80,483525.20
At-Risk / High-Value Churn,166,126.00,5.50,2228.00,369840.00
Lost / Hibernating,824,228.50,1.00,230.30,189770.90
Promising,188,37.50,1.50,1002.80,188522.30
New Inflow,234,18.80,1.10,242.60,56773.00


## 4. Pre-processing for clustering

Monetary and Frequency are heavily right-skewed. Log1p compresses the tail; StandardScaler puts all three axes on the same footing.

In [15]:
X_scaled, scaler, X_log = preprocess_for_clustering(rfm)
print("Scaled matrix shape:", X_scaled.shape)
print("Feature means after scaling (should be ~0):", X_scaled.mean(axis=0).round(4))

Scaled matrix shape: (4338, 3)
Feature means after scaling (should be ~0): [ 0. -0.  0.]


## 5. Choosing k — Elbow + Silhouette

In [16]:
wcss = elbow_method(X_scaled)
sil = silhouette_analysis(X_scaled)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Elbow (WCSS)", "Silhouette"])
fig.add_trace(go.Scatter(x=list(wcss.keys()), y=list(wcss.values()), mode="lines+markers", name="WCSS"), row=1, col=1)
fig.add_trace(go.Scatter(x=list(sil.keys()), y=list(sil.values()), mode="lines+markers", name="Silhouette"), row=1, col=2)
fig.update_layout(template="plotly_white", height=400, showlegend=False)
fig.show()

print("WCSS:", {k: round(v, 1) for k, v in wcss.items()})
print("Silhouette:", {k: round(v, 3) for k, v in sil.items()})

WCSS: {2: 6872.8, 3: 4298.6, 4: 3242.1, 5: 2760.9, 6: 2397.2, 7: 2166.7, 8: 1960.5, 9: 1818.9, 10: 1687.9}
Silhouette: {2: 0.406, 3: 0.416, 4: 0.38, 5: 0.343, 6: 0.331, 7: 0.3, 8: 0.301, 9: 0.281, 10: 0.282}


Silhouette peaks around k = 3–4 and the elbow softens in the same region. We adopt **k = 4**.

In [17]:
labels, model = run_kmeans(X_scaled, n_clusters=4)
rfm = attach_labels(rfm, labels, "Cluster_KMeans")
cluster_profiles(rfm, "Cluster_KMeans")

,Cluster_KMeans,N,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue
0,3,558,19.50,16.00,10007.50,5584207.90
1,0,1447,45.40,4.30,1667.70,2413118.00
2,1,1392,58.10,1.50,392.90,546901.20
3,2,941,259.20,1.40,390.20,367180.90


## 6. Hierarchical & DBSCAN comparison

In [18]:
rfm = attach_labels(rfm, run_hierarchical(X_scaled, 4), "Cluster_Hierarchical")
db_labels = run_dbscan(X_scaled, eps=0.9, min_samples=8)
rfm = attach_labels(rfm, db_labels, "Cluster_DBSCAN")

print("Hierarchical profile:")
print(cluster_profiles(rfm, "Cluster_Hierarchical"))
print("\nDBSCAN label counts (–1 = noise/outlier):")
print(rfm["Cluster_DBSCAN"].value_counts())

Hierarchical profile:
   Cluster_Hierarchical     N  Avg_Recency  Avg_Frequency  Avg_Monetary  \
0                     1  1367        32.30           6.60       2808.00   
1                     3   161         8.60          29.80      22560.40   
2                     0  1937        64.20           1.80        547.10   
3                     2   873       265.20           1.50        436.40   

   Total_Revenue  
0     3838478.60  
1     3632232.10  
2     1059751.20  
3      380946.00  

DBSCAN label counts (–1 = noise/outlier):
Cluster_DBSCAN
 0    4327
-1      11
Name: count, dtype: int64


## 7. 3-D view of the RFM space

In [19]:
sample = rfm.sample(min(2500, len(rfm)), random_state=42)
fig = px.scatter_3d(
    sample,
    x="Recency",
    y="Frequency",
    z="Monetary",
    color="Cluster_KMeans",
    opacity=0.65,
    log_y=True,
    log_z=True,
    title="K-Means clusters in RFM space (log scale on F & M)",
)
fig.update_layout(height=560, margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 8. Persist results

In [20]:
out = ROOT / "data"
rfm.to_csv(out / "rfm_segmented.csv", index=False)
joblib.dump(scaler, out / "rfm_scaler.joblib")
joblib.dump(model, out / "kmeans_model.joblib")
print("Saved rfm_segmented.csv + scaler + kmeans model")
print(rfm.shape)

Saved rfm_segmented.csv + scaler + kmeans model
(4338, 20)
